# Lesson 24 - Classical Machine Learning

**Goals:** Separate features/targets, split data, build leakage-resistant pipelines, and interpret regression and classification metrics.

**Before you begin:** Lessons 22–23 supply numerical arrays and labeled tables; functions and tests are already familiar. Previous stage: [23_pandas_data_analysis.ipynb](23_pandas_data_analysis.ipynb).
This lesson defines all its own inputs and runs independently on Python 3.11+.

**Setup:** `requirements.txt` (scikit-learn, pandas, NumPy). See the README shared requirements instructions. Install from a terminal only when you choose;
these notebooks never install packages. No core example needs a network, API key, database server, or download.
All records are fictional.

Run cells top to bottom with **Shift+Enter**. Predict the output before running each example,
change one input, rerun the relevant cells, and explain what changed. Restart the kernel and
Run All after restoring the original inputs. Practice cells intentionally contain no answers.
Attempt the exercises and project before opening `instructor_solutions/24_scikit_learn_classical_ml_solutions.ipynb`.


## 1. What supervised learning learns

A feature is an input available at prediction time; the target is the outcome to learn.
Regression predicts a number; classification predicts a category. We create energy from a
fictional relationship with occupied hours and temperature. The formula belongs to the data
generator, not to the model's input. IDs and future bills are not useful legitimate predictors.
`X` has shape `(samples, features)` and `y` has one target per sample.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix)

rng = np.random.default_rng(24)
hours = rng.uniform(0, 10, 120)
temperature = rng.uniform(15, 35, 120)
X = pd.DataFrame({"hours":hours, "temperature":temperature})
y = 5 + 3 * hours + 0.5 * temperature + rng.normal(0, 0.3, 120)
X.loc[::13, "temperature"] = np.nan  # Missing sensors, not missing targets.
print(X.head(3), y[:3], sep="\n")
assert X.shape == (120, 2) and y.shape == (120,)


## 2. Hold out unseen examples before learning

Training data fits the model. A held-out test set estimates behavior on unseen examples.
Use a fixed seed for a reproducible demonstration, not a search for a flattering split.
Random splitting suits these independent synthetic rows. For forecasting split chronologically;
for repeated employees/sites split by group to avoid related rows on both sides.
Never tune choices repeatedly against the test set.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=24)
print(X_train.shape, X_test.shape)
assert len(X_train) == 90 and len(X_test) == 30
assert set(X_train.index).isdisjoint(X_test.index)


## 3. Fit preprocessing inside a Pipeline

`fit` learns state, `transform` applies that state, and `predict` returns outputs.
Median imputation learns replacement values; scaling learns means and standard deviations.
Both must be fitted only on training data. A Pipeline sequences these steps and repeats exactly
the same transformations at prediction time. Linear regression fits a weighted sum of features.
Scaling is not essential for this unregularized regressor, but is useful in many other models.


In [ ]:
regression = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", LinearRegression()),
])
regression.fit(X_train, y_train)
predicted_kwh = regression.predict(X_test)
print(predicted_kwh[:3])
assert predicted_kwh.shape == (30,)
# Prediction rows use exactly the training columns, in the same order.
print(regression.predict(pd.DataFrame({"hours":[4.], "temperature":[25.]})))


## 4. Regression evaluation needs units and a baseline

MAE is average absolute error in target units (kWh). RMSE is the square root of mean
squared error and gives larger mistakes more weight. R² compares squared error to a constant
test-mean reference: 1 is perfect, 0 matches that reference, and negative is possible.
A DummyRegressor predicts the training mean and provides a practical baseline.
Do not promise a fixed score across package versions or altered data.


In [ ]:
baseline_regression = DummyRegressor(strategy="mean").fit(X_train, y_train)
mae = mean_absolute_error(y_test, predicted_kwh)
baseline_mae = mean_absolute_error(y_test, baseline_regression.predict(X_test))
print({"MAE_kwh": round(mae,3),
       "RMSE_kwh": round(float(np.sqrt(mean_squared_error(y_test, predicted_kwh))),3),
       "R2": round(r2_score(y_test, predicted_kwh),3),
       "baseline_MAE": round(baseline_mae,3)})
assert np.isfinite(predicted_kwh).all()
# Exact metric check independent of the fitted model.
assert mean_absolute_error([10,20], [12,18]) == 2
assert mean_squared_error([10,20], [12,18]) == 4


## 5. Classification predicts a class and a probability

Classify fictional high-use days. Logistic regression maps features to a class probability;
despite its name it is a classifier. `stratify` preserves class proportions approximately in
the split. Probability columns follow `classes_`; check their order. The default binary decision
threshold is 0.5. A probability is a model estimate, not a guarantee or proof of calibration.


In [ ]:
high_use = (y >= 33).astype(int)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X, high_use, test_size=0.25, random_state=24, stratify=high_use)
classifier = Pipeline([("impute", SimpleImputer(strategy="median")),
                       ("scale", StandardScaler()),
                       ("model", LogisticRegression(max_iter=300, random_state=24))])
classifier.fit(Xc_train, yc_train)
predicted_class = classifier.predict(Xc_test)
probabilities = classifier.predict_proba(Xc_test)
print(classifier.classes_, probabilities[:3])
assert probabilities.shape == (30, 2)
np.testing.assert_allclose(probabilities.sum(axis=1), 1)


## 6. Classification metrics answer different questions

Accuracy is the fraction correct. Precision asks how many predicted positives are real positives;
recall asks how many real positives were found. F1 balances precision and recall through their
harmonic mean. A confusion matrix with labels `[0,1]` has actual classes as rows and predicted
classes as columns: `[[TN,FP],[FN,TP]]`. Accuracy can hide rare-class failures.
We explicitly use `zero_division=0` when no positive predictions exist.


In [ ]:
baseline_classifier = DummyClassifier(strategy="most_frequent").fit(Xc_train, yc_train)
print(confusion_matrix(yc_test, predicted_class, labels=[0,1]))
print({"accuracy": accuracy_score(yc_test, predicted_class),
       "precision": precision_score(yc_test, predicted_class, zero_division=0),
       "recall": recall_score(yc_test, predicted_class, zero_division=0),
       "F1": f1_score(yc_test, predicted_class, zero_division=0),
       "baseline_accuracy": baseline_classifier.score(Xc_test, yc_test)})
known_actual, known_predicted = [0,0,1,1], [0,1,0,1]
assert confusion_matrix(known_actual, known_predicted).tolist() == [[1,1],[1,1]]
assert precision_score(known_actual, known_predicted) == 0.5
assert recall_score(known_actual, known_predicted) == 0.5


## 7. Make leakage visible and validate on training folds

Leakage uses information unavailable when a real prediction is made. Fitting a scaler on
the test set leaks its distribution even without labels. Including a future paid bill leaks
the outcome itself. A Pipeline prevents preprocessing leakage inside cross-validation too:
each fold fits its own imputer, scaler, and estimator. Cross-validation below uses training data
only. Use it to compare choices; evaluate your final choice on the test set once.
Change the extreme held-out value below and predict which mean changes.


In [ ]:
tiny_train = np.array([[0.], [2.]])
tiny_test = np.array([[100.]])
safe_scaler = StandardScaler().fit(tiny_train)
leaky_scaler = StandardScaler().fit(np.vstack([tiny_train, tiny_test]))  # Demonstration only.
print("Train-only mean:", safe_scaler.mean_, "leaked mean:", leaky_scaler.mean_)
assert safe_scaler.mean_[0] == 1
assert leaky_scaler.mean_[0] == 34
cv_mae = -cross_val_score(regression, X_train, y_train, cv=3, scoring="neg_mean_absolute_error")
print("Training-fold MAE:", np.round(cv_mae, 3))
assert cv_mae.shape == (3,)


## 8. Common mistakes and debugging

Check sample counts and row alignment before splitting. Never include the target or future-derived
columns in X. Preserve feature names/order at serving time. Read `NotFittedError` as a reminder to
fit first. Diagnose strange scores with a baseline, class counts, and a few prediction/error pairs.
Tiny synthetic examples teach mechanics; they do not establish real-world performance.
Categorical text needs encoding (for example OneHotEncoder in a ColumnTransformer), not arbitrary
integer codes implying order. That is a useful extension after the numerical workflow.


## Practice lab

Use the supplied inputs, add your own comments, and compare with the check values. Use `practice_`
names where helpful. Increase the difficulty one exercise at a time. Checks describe results;
they are not complete implementations. Test a boundary or malformed input when applicable.


### Exercise 1. Separate inputs from outcomes

From hours `[1,2,3,4]` and kWh `[8,11,14,17]`, create X and y.

**Check:** X `(4,1)`, y `(4,)`; kWh absent from X.


In [ ]:
practice_df = pd.DataFrame({"hours":[1,2,3,4], "kwh":[8,11,14,17]})
# TODO: choose feature columns and target.


### Exercise 2. Calculate known errors

For actual `[10,20,30]`, predicted `[12,18,33]`, calculate MAE and RMSE.

**Check:** MAE 7/3; RMSE sqrt(17/3).


In [ ]:
practice_actual = [10,20,30]
practice_predicted = [12,18,33]
# TODO: metrics and units.


### Exercise 3. Interpret alerts

Actual `[0,0,1,1,1]`, predicted `[0,1,1,0,1]`. Compute confusion matrix, precision, recall, F1.

**Check:** matrix `[[1,1],[1,2]]`; precision=recall=F1=2/3.


In [ ]:
practice_actual = [0,0,1,1,1]
practice_predicted = [0,1,1,0,1]
# TODO: metrics; explain a false negative.


### Exercise 4. Fit a small pipeline

Train imputer→linear regression on hours `[0,1,2,3]`, targets `[5,8,11,14]`. Predict hours 4.

**Check:** prediction approximately 17. Keep prediction input two dimensional.


In [ ]:
practice_X = np.array([[0.],[1.],[2.],[3.]])
practice_y = np.array([5.,8.,11.,14.])
# TODO: fit and predict without using a test target.


### Exercise 5. Prove preprocessing isolation

Fit an imputer→scaler→regression pipeline on `[[0],[2],[4]]`, targets `[1,5,9]`. Predict `[[100]]` and verify scaler mean does not change.

**Check:** mean 2 before and after; prediction 201. Explain why calling fit on combined data would invalidate the check.


In [ ]:
practice_train = np.array([[0.],[2.],[4.]])
practice_test = np.array([[100.]])
# TODO: fit training data only; inspect named_steps["scale"].mean_.


## Chapter project

Write a reusable regression experiment on 80 synthetic rows: hours are linspace(0,10,80),
target is `5 + 3*hours`. Split 25% with seed 24, fit a Pipeline, compare test MAE with a training-mean
DummyRegressor, and return a JSON-ready metrics dictionary. Keep raw data generation separate from training.

**Check:** 60 train/20 test rows, MAE < 1e-8 for the linear model and > 0 for the baseline.
Add a short model card describing units, features available at inference, split policy, limitations,
and why perfect synthetic scores do not imply perfect real-world predictions.


In [ ]:
def run_energy_experiment():
    # TODO: data, split, fit, baseline, metrics, model card.
    pass


## Reflection and knowledge check

1. What does fit learn that predict must reuse?
2. Why can imputation leak without target labels?
3. When is recall more useful than accuracy?
4. How should a forecasting split differ from this random split?

Choose one changed input, predict the new result, and explain your observation to a colleague.


## Official references

- [scikit-learn getting started](https://scikit-learn.org/stable/getting_started.html)
- [Pipelines](https://scikit-learn.org/stable/modules/compose.html)
